# Coursework 3

In [1]:
using JLD2, LinearAlgebra, Nemo, AbstractAlgebra


Welcome to Nemo version 0.57.2

Nemo comes with absolutely no warranty whatsoever


## 1. Your Data 

### Data Type Declaration

The data in the provided data file are stored using the data types defined below. These type definitions are required to load the data correctly.

**Do not modify these definitions.**

In [2]:
# Data type for Q12_1
#########################
# Binary Hamming Codes
#########################
mutable struct QBHamming
    p :: Int64 
    r :: Int64
    H :: Union{Missing,Array{Int64,2}}
    I :: Union{Missing,Vector{Int64}}
    G :: Union{Missing,Array{Int64,2}}
    Pe1 :: Float64
    Perr1 :: Union{Missing,Float64}
    y1 :: Vector{Int64}
    s1 :: Union{Missing,Vector{Int64}}
    c1 :: Union{Missing,Vector{Int64}}
    m1 :: Union{Missing,Vector{Int64}}
    Pe2 :: Float64
    Perr2 :: Union{Missing,Float64}
    y2 :: Vector{String}
    s2 :: Union{Missing,Vector{Int64}}
    c2 :: Union{Missing,Vector{Int64}}
    m2 :: Union{Missing,Vector{Int64}}
end

# Data type for Q13_1
#########################
# Matrix Inverse: Fq
#########################
mutable struct QMatInverse
    p :: Int
    irr_poly_coeff :: Vector{Int}
    A :: Matrix{Vector{Int}}
    Ainv :: Union{Missing,Matrix{Vector{Int}}}
end

# Data type for Q13_2
#####################################
# RS Codes on Fq
#####################################
mutable struct QRSFq
    p :: Int
    irr_poly_coeff :: Vector{Int}
    a_poly_coeff :: Vector{Int}
    G :: Union{Missing,Matrix{Vector{Int}}}
    H :: Union{Missing,Matrix{Vector{Int}}}
    y :: Vector{Vector{Int}}
    sv :: Union{Missing,Vector{Vector{Int}}}
    Lx :: Union{Missing,Vector{Vector{Int}}}
    Ex :: Union{Missing,Vector{Vector{Int}}}
    errL :: Union{Missing,Vector{Int}}
    errV :: Union{Missing,Vector{Vector{Int}}}
    xhat :: Union{Missing,Vector{Vector{Int}}}
    mv :: Union{Missing,Vector{Vector{Int}}}
end

### Load Your Data

In [3]:
filename = "CCT-CW3-06124635.jld2"

ans_Q12_1 = load(filename, "ans_Q12_1")
ans_Q13_1 = load(filename, "ans_Q13_1")
ans_Q13_2 = load(filename, "ans_Q13_2")


QRSFq(2, [1, 1, 0, 1, 1, 1], [0, 1, 0, 1], [[1] [1] … [1] [1]; [1] [0, 1, 0, 1] … [1, 1, 1, 1] [0, 0, 0, 0, 1]; … ; [1] [1, 1, 0, 1] … [0, 0, 1, 0, 1] [0, 1, 1]; [1] [1, 1] … [1, 1, 0, 1, 1] [0, 1, 1, 0, 1]], [[1] [0, 1, 0, 1] … [1, 1, 1, 1] [0, 0, 0, 0, 1]; [1] [1, 0, 0, 1] … [0, 0, 0, 1, 1] [1, 1, 1, 1]; … ; [1] [1, 0, 1] … [1, 1, 1] [1, 1, 0, 1, 1]; [1] [1, 0, 0, 1, 1] … [0, 1, 1, 0, 1] [0, 1, 1, 1, 1]], [[1, 0, 1], [1, 0, 1], [1, 0, 0, 1, 1], [0, 0, 0, 0, 1], [0, 1, 1, 1], [0, 1, 1, 1], [0, 0, 0, 1], [1, 1, 1], [0, 1, 1, 1], [0, 1, 0, 1]  …  [1, 0, 1, 1], [1], [1, 1, 1], [0, 1, 1, 1], [1, 0, 1, 1, 1], [1, 1, 1, 0, 1], [0, 0, 1, 1, 1], [1, 0, 0, 0, 1], [0, 0, 1, 1], [0, 1, 0, 1, 1]], [[1, 0, 1], [0, 0, 0, 0, 1], [1, 0, 0, 0, 1], [0, 0, 0, 1], [1, 1, 0, 0, 1], [0, 0, 0, 1, 1], Int64[], [1], [0, 0, 0, 0, 1], [0, 0, 1, 1, 1]], [[1], [1, 0, 1, 1, 1], [1], [0, 0, 0, 1, 1], [0, 0, 0, 1], [0, 0, 0, 1]], [[1, 0, 1], [1, 1, 1, 1], [0, 0, 1, 1, 1], [1, 1]], [8, 11, 15, 18, 22], [Int64[], Int6

## 2. Your Answers

### 12.1 Binary Hamming Codes

In [4]:
#--------------------------   1   --------------------------

# this function builds a binary Hamming parity-check matrix over F_p, where p is prime.
function parity_check_matrix(r::Int, p::Int)
    col = p^r-1
    H = Matrix{Int}(undef, r, col)

    for i in 1:col
        H[:,i] = digits(i; base = p, pad = r)
    end

    return H
end

    ans_Q12_1.H = parity_check_matrix(ans_Q12_1.r, ans_Q12_1.p)



#--------------------------   2   --------------------------

#this function returns the indices of columns of binary H containing exactly one 1
function parity_indeces(H::Matrix{Int})
    col = size(H, 2)
    
    #Input validation
    if col == 0
        throw(ArgumentError("H cannot be empty"))
    end

    J = Int[]
    
    for i in 1:col
        if sum(H[:,i]) == 1
            push!(J, i)
        end
    end

    return J
end

    ans_Q12_1.I = parity_indeces(ans_Q12_1.H)



#--------------------------   3   --------------------------

#This function returns the generator matrix
function generator_matrix(H::Matrix{Int}, J::Vector{Int})
    size_H = size(H)
    
    #Input validation
    if size_H[1] == 0
        throw(ArgumentError("H cannot be empty"))
    end
    
    if !(size(J)[1] == size_H[1])
        throw(ArgumentError("Invalid J"))
    end

    data_pos = setdiff(1:size_H[2], J)
    size_data = size(data_pos)[1]

    G = zeros(Int, size_data, size_H[2])
    for i in 1:size_data
        #Set the entry at position i to 1.
        G[i, data_pos[i]] = 1

        #For each row k of H, if H[k,data_pos[i]]=1, set the entry at position J[k] of the new row to 1.
        for k in 1:size_H[1]
            if H[k, data_pos[i]] == 1
                G[i, J[k]] = 1
            end
            
        end
    end
    return G
end

    ans_Q12_1.G = generator_matrix(ans_Q12_1.H, ans_Q12_1.I)



#--------------------------   4   --------------------------

#This function takes the probability that one bit flips p and the number of bits in a codeword n.
#It returns the decoding error probability
function decoding_error_probability(p::Float64, n::Int64)
    #Input validation
    if !(0<=p<=1)
        throw(ArgumentError("p must be a probability between 0 and 1"))
    end
    #Input validation
    if n < 3 || !ispow2(n + 1)
        throw(ArgumentError("n must be a Hamming code length: 2^r - 1 with r >= 2"))
    end

    pSuccess = 1-p

    zeroFlips = pSuccess^n
    oneFlips = n * p * pSuccess^(n-1)
    return 1-zeroFlips-oneFlips
end
        
    ans_Q12_1.Perr1 = decoding_error_probability(ans_Q12_1.Pe1, size(ans_Q12_1.H,2))



#--------------------------   5   --------------------------

#This function takes the parity-check matrix H and the received codeword y and returns the syndrome of the 
#received binary word y
function syndrome(H::Matrix{Int}, y::Vector{Int64})
    if length(y) != size(H, 2)
        throw(DimensionMismatch("y must have one bit per column of H"))
    end
        
    return mod.(H*y, 2)
end

    ans_Q12_1.s1 = syndrome(ans_Q12_1.H, ans_Q12_1.y1)

#this function returns the transmitted codeword supposing that only one flip flipped
function transmitted_codeword(y::Vector{Int64}, s::Vector{Int64})
    if length(y) != 2^length(s) - 1
        throw(DimensionMismatch("y must have 2^length(s) - 1 bits"))
    end
    
    if all(iszero, s)
        return copy(y)
    end
    
    bit = evalpoly(2, s)
    if !(1 <= bit <= length(y))
        throw(ArgumentError("s must identify a bit position in y"))
    end

    c = copy(y)
    c[bit] = mod(c[bit]+1, 2)
    return c
end

    ans_Q12_1.c1 = transmitted_codeword(ans_Q12_1.y1, ans_Q12_1.s1)
    data_pos = setdiff(1:length(ans_Q12_1.c1), ans_Q12_1.I)
    ans_Q12_1.m1 = ans_Q12_1.c1[data_pos]



#--------------------------   6   --------------------------

#This function takes the probability that one bit is lost p and the number of bits in a codeword n.
#It returns the decoding failure probability
function decoding_failure_probability(p::Float64, n::Int64)
    #Input validation
    if !(0<=p<=1)
        throw(ArgumentError("p must be a probability between 0 and 1"))
    end
    #Input validation
    if n < 3 || !ispow2(n + 1)
        throw(ArgumentError("n must be a Hamming code length: 2^r - 1 with r >= 2"))
    end

    pSuccess = 1-p

    zeroLost = pSuccess^n
    oneLost = n * p * pSuccess^(n-1)
    twoLost = binomial(n, 2) * p^2 * pSuccess^(n-2)
    return 1 - zeroLost - oneLost - twoLost
end
        
    ans_Q12_1.Perr2 = decoding_failure_probability(ans_Q12_1.Pe2, size(ans_Q12_1.H,2))



#--------------------------   7   --------------------------

function decode_hamming_erasures(y::Vector{String}, H::Matrix{Int})
    n = length(y)

    # Treat erasures as zeros so the syndrome reflects only the received bits.
    y_zeros = zeros(Int64, n)
    for i in 1:n
        if y[i] != "?"
            y_zeros[i] = parse(Int64, y[i])
        end
    end

    syn = syndrome(H, y_zeros)


    erased_pos = findall(==("?"), y)
    erased_pos_n = length(erased_pos)
    codeword = copy(y_zeros)
    
    if erased_pos_n == 0
        #y is the transmitted codeword
    
    elseif erased_pos_n == 1
        i = erased_pos[1]
        if all(iszero, syn)
            #codeword[i] = 0
        elseif syn == H[:,i]
            codeword[i] = 1
        else
           throw(ArgumentError("No bit matches the syndrome"))             
        end

    
    elseif erased_pos_n == 2
        i,j = erased_pos
    
        if all(iszero, syn)
             # Both erased bits were 0. Keep the zeros already in codeword.
        elseif (syn == H[:, i])
            codeword[i] = 1
        elseif (syn == H[:, j])
            codeword[j] = 1
        elseif syn == mod.(H[:, i] + H[:, j], 2)
            codeword[i] = 1
            codeword[j] = 1
        else
           throw(ArgumentError("No bit pair matches the syndrome"))             
        end
    
    else
        throw(ArgumentError("There must be maximum two erasures"))
    end


    # The systematic generator matrix places the message bits at the non-parity positions.
    data_pos = setdiff(1:n, parity_indeces(H))
    message = codeword[data_pos]

    return syn, codeword, message
end

    ans_Q12_1.s2, ans_Q12_1.c2, ans_Q12_1.m2 = decode_hamming_erasures(ans_Q12_1.y2, ans_Q12_1.H)

([1, 0, 0, 0, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 1, 0  …  0, 0, 0, 0, 1, 1, 0, 1, 0, 1], [0, 0, 1, 0, 1, 0, 1, 1, 1, 0  …  0, 0, 0, 0, 1, 1, 0, 1, 0, 1])

### 13.1 Matrix Inverse: Fq

In [5]:
# This function builds the finite field F_q = F_p[u]/(f(u)) from p and the coefficient vector of f
# (f is given in ascending order [a0, a1, ..., ad])
function buildField(p::Integer, fcoeffs::AbstractVector{<:Integer})
    Fp = Nemo.GF(p)
    R, u = polynomial_ring(Fp, "u") 
    fpoly = R(fcoeffs)
    if !is_irreducible(fpoly)
        error("f is not irreducible over F_$p")
    end
    F, α = finite_field(fpoly, "α")
    return F, α, degree(fpoly)
end

# This function transforms the integer coefficient vector v into an element of the finite field F_q with generator α
function vecToField(F, α, v::AbstractVector{<:Integer})
    res = zero(F)
    
    for i in eachindex(v)
        res += F(v[i]) * α^(i-1)
    end
    
    return res
end

# This function transforms an element of F_q into its integer coefficient vector [a0, a1, ..., a_deg]
# with coefficients in {0, ..., p-1} 
function fieldToVec(x, d::Integer)
    p = Int(characteristic(parent(x)))             # p of the field F_q
    ZZw, w = Nemo.polynomial_ring(Nemo.ZZ, "w")
    xZZ = lift(ZZw, x)
    return mod.(Int.(collect(coefficients(xZZ))), p)
end

# This function transforms an integer-vector matrix into a Nemo matrix over the finite field F_q
function matrixToField(F, α, A::AbstractMatrix)
    m, n = size(A)
    M = zero_matrix(F, m, n)                # Nemo zero matrix over F
    for i in 1:m                            # rows
        for j in 1:n                        # columns
            M[i, j] = vecToField(F, α, A[i, j])
        end
    end
    return M
end

# This function transforms a Nemo matrix over F_q into an integer-vector matrix
function fieldToMatrix(M::MatElem, d::Integer)
    m, n = size(M)
    A = Matrix{Vector{Int}}(undef, m, n)    # Julia matrix whose entries are integer vectors
    for i in 1:m                            # rows
        for j in 1:n                        # columns
            A[i, j] = fieldToVec(M[i, j], d)
        end
    end
    return A
end

#This function takes a square matrix A over a field and returns its adjugate A*
function adjugate(A::MatElem)
    
    #Input validation: check if the matrix is square
    dim = size(A,1)
    if (dim != size(A,2))
        error("Input is not a square matrix")
    end

    #Initialize the square matrix of the same dimensions
    F = base_ring(A)  #the field of A
    adj = zero_matrix(F, dim, dim)

    
    for i in 1 : dim
        for j in 1: dim
            #Compute exact cofactors using submatrices
            adj[i, j] = (-1)^(i + j) * det(A[setdiff(1:dim, j), setdiff(1:dim, i)])
        end
    end
    
    return adj
end


#This function computes the inverse of a square matrix A over a finite field 𝔽_q
#A^(-1)=Adj(A)/det(A) 
function invMatrixFq(A::MatElem)
    
    #Input validation: check if A is a square matrix
    dim = size(A,1)
    if (dim != size(A,2))
        error("Input A is not a square matrix")
    end

    determinant = det(A)

     # In a field, A is invertible if and only if det(A) ≠ 0
    if !iszero(determinant)
        return inv(determinant) * adjugate(A) 
    else
        println("The matrix")
        display(A)
        println("is not invertible over F_q(det=0)")
        return zero_matrix(base_ring(A), 0, 0)
    end
end



    F, α, d = buildField(ans_Q13_1.p, ans_Q13_1.irr_poly_coeff)
    A_F = matrixToField(F, α, ans_Q13_1.A)
    Ainv_F = invMatrixFq(A_F)
    @assert A_F * Ainv_F == identity_matrix(F, nrows(A_F))   # verification

    ans_Q13_1.Ainv = fieldToMatrix(Ainv_F, d) 

    

5×5 Matrix{Vector{Int64}}:
 [0, 1, 1, 2]  [0, 2, 2]     [1, 0, 2, 2]  [0, 1, 2, 2]  [1, 0, 1, 2]
 [0, 2, 1, 1]  [2, 1, 2]     [1, 2, 2]     [0, 1, 0, 1]  [1, 2, 2, 1]
 [2, 2]        [1, 2, 1, 2]  [0, 1, 0, 1]  [0, 1, 2, 2]  [2, 0, 0, 1]
 [2, 0, 2, 2]  [1, 2, 1, 1]  [1, 0, 0, 1]  [2, 2, 0, 2]  [2, 2, 2, 2]
 [1, 1, 1, 2]  [0, 0, 2, 1]  [0, 2, 0, 1]  [0, 0, 2, 1]  [1, 0, 1, 2]

### 13.2 RS Codes on Fq

In [6]:
#--------------------------   1   --------------------------

# This function builds the generator matrix G and the parity-check matrix H
# of the [n, k] Reed-Solomon code with defining points {1, α, ..., α^(n-1)}
function rsMatrices(α, n::Integer, k::Integer)

    # Input validation
    if !(0 < k < n)
        error("Invalid parameters: need 0 < k < n")
    end

    F = parent(α)                      # the field α belongs to (F_32)

    # Generator matrix: k × n, G[i, j] = α^((i-1)(j-1))
    G = zero_matrix(F, k, n)
    for i in 1:k
        for j in 1:n
            G[i, j] = α^((i - 1) * (j - 1))
        end
    end

    # Parity-check matrix: (n-k) × n, H[i, j] = α^(i(j-1))
    H = zero_matrix(F, n - k, n)
    for i in 1:(n - k)
        for j in 1:n
            H[i, j] = α^(i * (j - 1))
        end
    end

    return G, H
end

    F32, a, d32 = buildField(ans_Q13_2.p, ans_Q13_2.irr_poly_coeff)
    α = vecToField(F32, a, ans_Q13_2.a_poly_coeff)

    G, H = rsMatrices(α, 31, 21)

    @assert α != one(F32) && α^31 == one(F32)
    @assert iszero(G * transpose(H))

    ans_Q13_2.G = fieldToMatrix(G, d32)
    ans_Q13_2.H = fieldToMatrix(H, d32)

#--------------------------   2   --------------------------

# This function transforms a vector of integer coefficient vectors into a vector of elements of F_q
function vectorToField(F, a, V::AbstractVector)
    res = elem_type(F)[]
    for v in V
        push!(res, vecToField(F, a, v))
    end
    return res
end

# This function transforms a vector of elements of F_q into a vector of integer coefficient vectors
function fieldToVector(X::AbstractVector, d::Integer)
    res = Vector{Int}[]
    for x in X
        push!(res, fieldToVec(x, d))
    end
    return res
end

# This function computes the syndrome s = y·H^(T) of a received word y over F_q
function rsSyndrome(y::AbstractVector, H::MatElem)
    n = size(H,2)

    # Input validation
    if length(y) != n
        error("y must have one symbol per column of H")
    end

    F = base_ring(H)
    y_row = matrix(F, 1, n, y)          # y as a 1×n row matrix
    s_row = y_row * transpose(H)        # 1×(n-k)

    res = elem_type(F)[]
    for j in 1:nrows(H)
        push!(res, s_row[1, j])
    end
    
    return res
end

    y_F = vectorToField(F32, a, ans_Q13_2.y)     # y in the field
    s   = rsSyndrome(y_F, H)                     

    ans_Q13_2.sv = fieldToVector(s, d32)


#--------------------------   3   --------------------------

# This function solves the key equation L(x)s(x) ≡ E(x) mod x^(2t) using the extended Euclidean algorithm, stopped when deg(r) < t.
# It returns the locator L(x) (with L(0) = 1) and the evaluator E(x).
function keyEquation(s_poly, t::Integer)
    Fx = parent(s_poly)               # the polynomial ring F_q[x]
    x  = gen(Fx)                      # its variable

    r_prev, r_curr = x^(2t), s_poly   # r_{-1}, r_0
    t_prev, t_curr = zero(Fx), one(Fx)  # t_{-1}, t_0

    while degree(r_curr) >= t
        q, r_new = divrem(r_prev, r_curr)
        t_new = t_prev - q * t_curr 
        
        r_prev = r_curr
        r_curr = r_new
        t_prev = t_curr
        t_curr = t_new
    end

    c = coeff(t_curr, 0)              # normalisation constant
    return t_curr * inv(c), r_curr * inv(c)
end

    Fx, x = Nemo.polynomial_ring(F32, "x")
    s_poly = Fx(s)                      

    L, E = keyEquation(s_poly, 5)

    @assert coeff(L, 0) == one(F32)              # L(0) = 1
    @assert degree(L) <= 5 && degree(E) < 5      # degree conditions
    @assert iszero(rem(L * s_poly - E, x^10))    # key equation

    ans_Q13_2.Lx = [fieldToVec(coeff(L, i), d32) for i in 0:degree(L)]
    ans_Q13_2.Ex = [fieldToVec(coeff(E, i), d32) for i in 0:degree(E)]


#--------------------------   4   --------------------------

# This function finds the error locations by testing L(α^(-i)) = 0 for i = 0, ..., n-1
# It returns the positions indexed from 1 to n.
function errorLocations(L, α, n::Integer)
    locations = Int[]
    αinv = inv(α)

    for i in 0:(n - 1)
        if iszero(L(αinv^i))          # L evaluated at α^(-i)
            push!(locations, i + 1)   # positions are indexed from 1
        end
    end

    return locations
end

    errL = errorLocations(L, α, 31)

    @assert length(errL) == degree(L)     # one root per error
    println("Error locations: ", errL)

    ans_Q13_2.errL = errL


#--------------------------   5   --------------------------

# This function computes the error vector e using Forney's formula e_i = -E(α^(-i)) / L'(α^(-i))   
# errL contains the error positions indexed from 1 to n.
function errorVector(L, E, α, errL::Vector{Int}, n::Integer)
    F = parent(α)
    e = [zero(F) for _ in 1:n]        # vector of n zeros of the field
    dL = derivative(L)
    αinv = inv(α)

    for pos in errL
        i = pos - 1                   # back to the 0-based index of the notes
        X = αinv^i                    # α^(-i)
        e[pos] = -E(X) * inv(dL(X))
    end

    return e
end

    e = errorVector(L, E, α, errL, 31)

    # Verifications
    @assert all(!iszero(e[pos]) for pos in errL)    # nonzero values at the error positions
    @assert rsSyndrome(e, H) == s                    # e must have the same syndrome as y

    ans_Q13_2.errV = fieldToVector(e, d32)


#--------------------------   6   --------------------------

# This function returns the estimated transmitted codeword x_hat = y - e
function transmittedCodeword(y::AbstractVector, e::AbstractVector)
    if length(y) != length(e)
        error("y and e must have the same length")
    end
    return y .- e                     # componentwise subtraction in F_q
end

    xhat = transmittedCodeword(y_F, e)

    # Verifications
    @assert all(iszero, rsSyndrome(xhat, H))                     # x_hat is a codeword: x_hat·H^T = 0
    @assert findall(xhat .!= y_F) == errL                         # x_hat differs from y only at the error positions

    ans_Q13_2.xhat = fieldToVector(xhat, d32)


#--------------------------   7   --------------------------

# This function recovers the message m from the codeword x_hat = m·G,
# using the first k columns of G 
function rsMessage(xhat::AbstractVector, G::MatElem)
    k, n = nrows(G), ncols(G)

    # Input validation
    if length(xhat) != n
        error("xhat must have one symbol per column of G")
    end

    F = base_ring(G)
    Gk = G[1:k, 1:k]                          # first k columns: k×k Vandermonde
    xk = matrix(F, 1, k, xhat[1:k])           # first k symbols of x_hat as a row

    m_row = xk * invMatrixFq(Gk)              

    return [m_row[1, j] for j in 1:k]
end

    m = rsMessage(xhat, G)

    # Verifications
    @assert length(m) == 21
    @assert matrix(F32, 1, 21, m) * G == matrix(F32, 1, 31, xhat)   # m·G must give back x̂

    ans_Q13_2.mv = fieldToVector(m, d32)


Error locations: [8, 11, 15, 18, 22]


21-element Vector{Vector{Int64}}:
 [1, 0, 1, 1]
 [1, 1, 0, 1]
 [0, 1, 0, 1]
 [1, 0, 1, 1]
 [0, 0, 1, 1]
 []
 [1, 1, 1, 1]
 [0, 0, 1, 1, 1]
 [0, 0, 0, 1]
 [0, 0, 1, 0, 1]
 [0, 1, 0, 0, 1]
 [1, 0, 0, 1]
 [0, 1, 0, 1, 1]
 [1, 1, 1]
 [0, 0, 1, 1]
 [0, 1, 1, 0, 1]
 [1, 1, 1, 0, 1]
 [0, 0, 0, 1, 1]
 [0, 1, 0, 1]
 [0, 1, 1, 1]
 [0, 0, 0, 0, 1]

## 3. Save All Your Answers

Remember to save your answers into your data file!

In [7]:
jldsave(filename;
     ans_Q12_1, ans_Q13_1, ans_Q13_2
 );